# Prepare the signed 75,000-row trial

Derives 75,000 training, 10,000 validation, and 20,000 test rows from the already prepared 900k cohort. No dataset download or full split rebuild is performed.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
SOURCE_EXPERIMENT_NAME='lending_club_900k'
TRIAL_EXPERIMENT_NAME='lending_club_75k_trial'
FORCE_REBUILD_TRIAL=False
RUN_TESTS=True
# ==================================================================


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive'); repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir(): subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else: subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
src_path=str(repo/'src')
if src_path not in sys.path: sys.path.insert(0,src_path)
import tfmbench
print('tfmbench package:',tfmbench.__file__)
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
from tfmbench.config import BenchmarkConfig
from tfmbench.split import verify_manifest
source_config=BenchmarkConfig(DRIVE_OUTPUT_DIR,SOURCE_EXPERIMENT_NAME); source_config.validate()
source_manifest=verify_manifest(source_config.shared_dir,source_config)
print('Verified source manifest:',source_manifest['manifest_hash'])


In [ ]:
from tfmbench.trial import prepare_trial
result=prepare_trial(DRIVE_OUTPUT_DIR,SOURCE_EXPERIMENT_NAME,TRIAL_EXPERIMENT_NAME,force_rebuild=FORCE_REBUILD_TRIAL)
print('Trial folder:',result.experiment_root)
print('Trial manifest:',result.manifest['manifest_hash'])


In [ ]:
import pandas as pd
display(pd.DataFrame(result.manifest['splits']).T)
display(result.leakage_report)
print('Next notebook: 07_lightgbm_75k.ipynb')
